# Policy Training with MuJoCo Warp

This notebook provides a skeleton for training a reinforcement learning policy using the `stretch_mujoco_warp` model and MuJoCo Warp in Python. MuJoCo Warp (MJWarp) enables high-throughput, parallelized physics simulations directly on NVIDIA GPUs, drastically reducing RL model training time.

In [ ]:
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.5'
!uv pip install mujoco-warp
!uv pip install -U "jax[cuda12]" optax flax
!uv pip install numpy==2.2.5


In [ ]:
import os
import subprocess
# import mediapy as media
import numpy as np
import warp as wp

# Set up GPU rendering.
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Configuring MuJoCo for GPU rendering, setting ', end="")
%env MUJOCO_GL=egl

# avoid warp output in cells
wp.config.quiet = True



## 1. Load the Stretch Environment
We dynamically generate the MJCF using `Stretch4MujocoSimulator.get_robot_xml_path()`, which generates the URDF and parses it. It then creates the `stretch_4_dynamic.xml` which replaces the `.STL` collision meshes with solid primitives for efficient GPU evaluation. We then load the default scene that includes the dynamically generated robot XML.

In [ ]:
NWORLD = 16
num_iterations = 20 
rollout_length = 128
batch_size = NWORLD * rollout_length 
minibatch_size = 4096  

# NWORLD = 1024
# num_iterations = 2000 
# rollout_length = 128
# batch_size = NWORLD * rollout_length 
# minibatch_size = 4096    

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import mujoco
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import warp as wp
import numpy as np

try:
    import mujoco_warp as mjw
except ImportError:
    print("mujoco_warp not installed!")

wp.init()

from stretch_mujoco.robocasa_gen import model_generation_wizard
from stretch_mujoco.stretch4_mujoco_simulator import Stretch4MujocoSimulator

# 1. Generate the robot XML dynamically
robot_xml_path = Stretch4MujocoSimulator.get_robot_xml_path()

# 2. Use Robocasa to generate the scene
model, xml, objects_info = model_generation_wizard(
    stretch_xml_absolute=robot_xml_path,
    layout=1,
    style=1,
    task="PickPlaceCounterToCabinet",
    objects_list=["apple"],
)

# Display available objects related to the chosen object category
print("Available objects:", objects_info)
print("Available objects matching 'apple':")
target_object_body_name = None
for body_name, info in objects_info.items():
    if "apple" in info["cat"]:
        print(f" - {body_name} (Category: {info['cat']})")
        if target_object_body_name is None:
            target_object_body_name = body_name

if target_object_body_name is None:
    target_object_body_name = "obj_main" # Fallback
print(f"Selected target object for training: {target_object_body_name}")

In [ ]:
# 3. Disable unsupported solver options
model.opt.noslip_iterations = 0
model.opt.integrator = mujoco.mjtIntegrator.mjINT_EULER

# Parse into MjModel
warp_model = mjw.put_model(model)
print("Model loaded successfully. Nu:", model.nu, "Nq:", model.nq, "Nv:", model.nv)

print(f"Initializing {NWORLD} environments for training.")
# Pass nconmax and njmax manually just to be safe
data = mjw.make_data(model, nworld=NWORLD, nconmax=512, njmax=512)
mjw.forward(warp_model, data)

global_mj_model = model
global_warp_model = warp_model
global_warp_data = data

# Define the PPO Networks and Loss Functions

def sample_normal(rng, mean, log_std):
    std = jnp.exp(log_std)
    return mean + std * jax.random.normal(rng, mean.shape)

def log_prob_normal(x, mean, log_std):
    std = jnp.exp(log_std)
    var = std ** 2
    log_scale = log_std + 0.5 * jnp.log(2.0 * jnp.pi)
    return -((x - mean) ** 2) / (2.0 * var) - log_scale

class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, x):
        # Actor
        a = nn.Dense(256)(x)
        a = nn.relu(a)
        a = nn.Dense(256)(a)
        a = nn.relu(a)
        actor_mean = nn.Dense(self.action_dim)(a)
        
        actor_log_std = self.param("log_std", nn.initializers.zeros, (self.action_dim,))
        
        # Critic
        c = nn.Dense(256)(x)
        c = nn.relu(c)
        c = nn.Dense(256)(c)
        c = nn.relu(c)
        critic = nn.Dense(1)(c)
        
        return actor_mean, actor_log_std, jnp.squeeze(critic, -1)

@jax.jit
def get_action_and_value(params, x, rng):
    mean, log_std, value = ActorCritic(global_mj_model.nu).apply(params, x)
    action = sample_normal(rng, mean, log_std)
    log_prob = log_prob_normal(action, mean, log_std).sum(-1)
    return action, log_prob, value

@jax.jit
def get_value(params, x):
    _, _, value = ActorCritic(global_mj_model.nu).apply(params, x)
    return value

@jax.jit
def compute_gae(rewards, values, next_value, dones, gamma=0.99, gae_lambda=0.95):
    # Calculate GAE for a single rollout trajectory
    advantages = jnp.zeros_like(rewards)
    lastgaelam = jnp.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            nextnonterminal = 1.0 - dones[t]
            nextvalues = next_value
        else:
            nextnonterminal = 1.0 - dones[t]
            nextvalues = values[t + 1]
        delta = rewards[t] + gamma * nextvalues * nextnonterminal - values[t]
        lastgaelam = delta + gamma * gae_lambda * nextnonterminal * lastgaelam
        advantages = advantages.at[t].set(lastgaelam)
    returns = advantages + values
    return advantages, returns

@jax.jit
def update_ppo(params, opt_state, obs, actions, log_probs_old, returns, advantages):
    def loss_fn(p):
        mean, log_std, value = ActorCritic(global_mj_model.nu).apply(p, obs)
        log_probs = log_prob_normal(actions, mean, log_std).sum(-1)
        
        ratio = jnp.exp(log_probs - log_probs_old)
        surr1 = ratio * advantages
        surr2 = jnp.clip(ratio, 1.0 - 0.2, 1.0 + 0.2) * advantages
        
        actor_loss = -jnp.minimum(surr1, surr2).mean()
        critic_loss = jnp.mean((returns - value) ** 2)
        entropy_loss = jnp.mean(log_std + 0.5 + 0.5 * jnp.log(2 * jnp.pi))
        
        total_loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy_loss
        return total_loss, (actor_loss, critic_loss, entropy_loss)
    
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, (al, cl, el)), grads = grad_fn(params)
    updates, opt_state = tx.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, opt_state, loss

def get_obs_from_warp(global_warp_data):
    # observation = concat [qpos, qvel, gripper_pos, object_pos]
    qpos = wp.to_jax(global_warp_data.qpos)
    qvel = wp.to_jax(global_warp_data.qvel)
    xpos = wp.to_jax(global_warp_data.xpos)
    
    # Extract gripper and object positions
    robot_gripper_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, "link_grasp_center")
    object_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, target_object_body_name)
    
    gripper_pos = xpos[:, robot_gripper_id, :]
    object_pos = xpos[:, object_id, :]
    
    return jnp.concatenate([qpos, qvel, gripper_pos, object_pos], axis=-1)

def compute_rewards(obs):
    alive_bonus = 1.0
    # The first nq+nv elements are qpos and qvel
    vel_penalty = -0.01 * jnp.sum(jnp.square(obs[:, global_mj_model.nq:global_mj_model.nq+global_mj_model.nv]), axis=-1)
    
    # gripper_pos is at index [-6:-3], object_pos is at [-3:]
    gripper_pos = obs[:, -6:-3]
    object_pos = obs[:, -3:]
    
    dist = jnp.linalg.norm(gripper_pos - object_pos, axis=-1)
    # Give a negative reward based on distance to the object
    dist_reward = -10.0 * dist
    
    return alive_bonus + vel_penalty + dist_reward

def env_step(jax_actions, global_warp_data, rng):
    wp_actions = wp.from_jax(jax_actions, dtype=wp.float32)
    global_warp_data.ctrl = wp_actions
    for _ in range(4):
        mjw.step(global_warp_model, global_warp_data)
    obs = get_obs_from_warp(global_warp_data)
    rewards = compute_rewards(obs)
    rng, reset_rng = jax.random.split(rng)
    resets = jax.random.bernoulli(reset_rng, p=0.01, shape=(NWORLD,))
    return obs, rewards, resets, rng

rng = jax.random.PRNGKey(42)
rng, init_rng = jax.random.split(rng)
dummy_obs = jnp.zeros((NWORLD, global_mj_model.nq + global_mj_model.nv + 6))
params = ActorCritic(global_mj_model.nu).init(init_rng, dummy_obs)

tx = optax.adam(learning_rate=3e-4)
opt_state = tx.init(params)

print("Starting training...")
obs = get_obs_from_warp(global_warp_data)

for i in range(num_iterations):
    all_obs = []
    all_actions = []
    all_rewards = []
    all_dones = []
    all_log_probs = []
    all_values = []
    
    for step in range(rollout_length):
        rng, action_rng = jax.random.split(rng)
        actions, log_probs, values = get_action_and_value(params, obs, action_rng)
        
        next_obs, rewards, dones, rng = env_step(actions, global_warp_data, rng)
        
        all_obs.append(obs)
        all_actions.append(actions)
        all_rewards.append(rewards)
        all_dones.append(dones)
        all_log_probs.append(log_probs)
        all_values.append(values)
        
        obs = next_obs
        
    all_obs = jnp.stack(all_obs)
    all_actions = jnp.stack(all_actions)
    all_rewards = jnp.stack(all_rewards)
    all_dones = jnp.stack(all_dones)
    all_log_probs = jnp.stack(all_log_probs)
    all_values = jnp.stack(all_values)
    
    next_value = get_value(params, obs)
    advantages, returns = compute_gae(all_rewards, all_values, next_value, all_dones)
    
    b_obs = all_obs.reshape(-1, all_obs.shape[-1])
    b_actions = all_actions.reshape(-1, all_actions.shape[-1])
    b_log_probs = all_log_probs.reshape(-1)
    b_returns = returns.reshape(-1)
    b_advantages = advantages.reshape(-1)
    
    b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
    
    params, opt_state, loss = update_ppo(
        params, opt_state, b_obs, b_actions, b_log_probs, b_returns, b_advantages
    )
    
    print(f"Iteration {i}, Mean Reward: {all_rewards.mean():.4f}, Loss: {loss:.4f}")
    
print("Training finished!")



In [ ]:
# Save the trained parameters
from flax import serialization

# Convert parameters to bytes
bytes_output = serialization.to_bytes(params)

# Save to disk
with open("trained_ppo_params.msgpack", "wb") as f:
    f.write(bytes_output)
    
print("Saved trained model parameters to trained_ppo_params.msgpack")


In [ ]:
# Load the trained model and run it in the standard MuJoCo Viewer
import mujoco
import mujoco.viewer
import time
import jax.numpy as jnp
import numpy as np
from flax import serialization

# Load from disk
with open("trained_ppo_params.msgpack", "rb") as f:
    bytes_input = f.read()

loaded_params = serialization.from_bytes(params, bytes_input)

# Create a fresh MjData for standard simulation
mj_data = mujoco.MjData(global_mj_model)

# Reset environment
mujoco.mj_resetData(global_mj_model, mj_data)

print("Launching MuJoCo Viewer...")
try:
    with mujoco.viewer.launch_passive(global_mj_model, mj_data) as viewer:
        for step in range(1000):
            if not viewer.is_running():
                break
                
            # Construct observation
            qpos = jnp.array(mj_data.qpos)
            qvel = jnp.array(mj_data.qvel)
            robot_gripper_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, "link_grasp_center")
            object_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, target_object_body_name)
            gripper_pos = jnp.array(mj_data.xpos[robot_gripper_id])
            object_pos = jnp.array(mj_data.xpos[object_id])
            obs = jnp.concatenate([qpos, qvel, gripper_pos, object_pos], axis=-1)
            
            # Get deterministic action from the policy (mean of the distribution)
            action_mean, _, _ = ActorCritic(global_mj_model.nu).apply(loaded_params, obs)
            
            # Apply to simulation
            mj_data.ctrl[:] = np.array(action_mean)
            mujoco.mj_step(global_mj_model, mj_data)
            
            viewer.sync()
            time.sleep(global_mj_model.opt.timestep)
except Exception as e:
    print("Viewer closed or error:", e)


In [ ]:
# Load the trained model and render a video in the notebook
import mujoco
import mediapy as media
import jax.numpy as jnp
import numpy as np
from flax import serialization

# Load from disk
with open("trained_ppo_params.msgpack", "rb") as f:
    bytes_input = f.read()

loaded_params = serialization.from_bytes(params, bytes_input)

# Create a fresh MjData for standard simulation
mj_data = mujoco.MjData(global_mj_model)

# Reset environment
mujoco.mj_resetData(global_mj_model, mj_data)

# Create a renderer
renderer = mujoco.Renderer(global_mj_model, 480, 640)

print("Simulating and rendering...")
frames = []
for step in range(500): # Render 500 frames
    # Construct observation
    qpos = jnp.array(mj_data.qpos)
    qvel = jnp.array(mj_data.qvel)
    robot_gripper_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, "link_grasp_center")
    object_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, target_object_body_name)
    gripper_pos = jnp.array(mj_data.xpos[robot_gripper_id])
    object_pos = jnp.array(mj_data.xpos[object_id])
    obs = jnp.concatenate([qpos, qvel, gripper_pos, object_pos], axis=-1)
    
    # Get deterministic action from the policy
    action_mean, _, _ = ActorCritic(global_mj_model.nu).apply(loaded_params, obs)
    
    # Apply to simulation
    mj_data.ctrl[:] = np.array(action_mean)
    mujoco.mj_step(global_mj_model, mj_data)
    
    # Render frame every 10 steps (for 30fps assuming 0.002s timestep)
    if step % 10 == 0:
        renderer.update_scene(mj_data, camera="robot0_robotview")
        frames.append(renderer.render())

# Display the video in the notebook
media.show_video(frames, fps=30)



In [ ]:
# Visualize parallelized training worlds in a grid using mujoco-warp
import mujoco
import mediapy as media
import jax.numpy as jnp
import numpy as np
import warp as wp
import mujoco_warp as mjw
from flax import serialization

# Define grid size
GRID_W, GRID_H = 4, 4
BATCH_NWORLD = GRID_W * GRID_H
CAM_RES = (160, 120)  # Width, Height

# Load the trained model from disk
with open("trained_ppo_params.msgpack", "rb") as f:
    bytes_input = f.read()
loaded_params = serialization.from_bytes(params, bytes_input)

# Create a fresh batched data
batch_data = mjw.make_data(global_mj_model, nworld=BATCH_NWORLD, nconmax=512, njmax=512)

# Randomize the initial qpos slightly to show diversity
qpos = batch_data.qpos.numpy()
# Let's just slightly perturb the starting positions
qpos[:, 0] += np.random.uniform(-0.1, 0.1, size=BATCH_NWORLD) # x
qpos[:, 1] += np.random.uniform(-0.1, 0.1, size=BATCH_NWORLD) # y
batch_data.qpos = wp.array(qpos, dtype=wp.float32)
mjw.forward(global_warp_model, batch_data)

# Create warp render context
rc = mjw.create_render_context(
    global_mj_model,
    nworld=BATCH_NWORLD,
    cam_res=CAM_RES,
    render_rgb=True,
    render_depth=False,
    render_seg=False,
)

print(f"Simulating and rendering {BATCH_NWORLD} worlds in parallel...")
grid_frames = []

cam_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_CAMERA, "robot0_robotview")
# If camera not found, fallback to default free camera (-1)
if cam_id == -1:
    cam_id = -1

for step in range(300): # Render 300 steps
    # Construct observation (batched)
    qpos_jnp = jnp.array(batch_data.qpos.numpy())
    qvel_jnp = jnp.array(batch_data.qvel.numpy())
    xpos_jnp = jnp.array(batch_data.xpos.numpy())
    
    robot_gripper_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, "link_grasp_center")
    object_id = mujoco.mj_name2id(global_mj_model, mujoco.mjtObj.mjOBJ_BODY, target_object_body_name)
    
    gripper_pos = xpos_jnp[:, robot_gripper_id, :]
    object_pos = xpos_jnp[:, object_id, :]
    obs = jnp.concatenate([qpos_jnp, qvel_jnp, gripper_pos, object_pos], axis=-1)
    
    # Get deterministic action from the policy for the batch
    action_mean, _, _ = ActorCritic(global_mj_model.nu).apply(loaded_params, obs)
    
    # Apply to simulation
    batch_data.ctrl = wp.array(np.array(action_mean), dtype=wp.float32)
    mjw.step(global_warp_model, batch_data)
    
    # Render frame every 10 steps
    if step % 10 == 0:
        mjw.refit_bvh(global_warp_model, batch_data, rc)
        mjw.render(global_warp_model, batch_data, rc)
        
        rgb_data = wp.zeros((BATCH_NWORLD, CAM_RES[1], CAM_RES[0]), dtype=wp.vec3)
        mjw.get_rgb(rc, camera_index=cam_id, rgb_out=rgb_data)
        
        # Display our rendered worlds in a grid
        rgb_grid = rgb_data.numpy().reshape(GRID_H, GRID_W, CAM_RES[1], CAM_RES[0], 3)
        rgb_grid = rgb_grid.transpose(0, 2, 1, 3, 4)
        rgb_grid = rgb_grid.reshape(GRID_H * CAM_RES[1], GRID_W * CAM_RES[0], 3)
        
        # Convert float RGB to uint8
        if rgb_grid.dtype != np.uint8:
            if rgb_grid.max() <= 1.0:
                rgb_grid = (rgb_grid * 255).astype(np.uint8)
            else:
                rgb_grid = rgb_grid.astype(np.uint8)
                
        grid_frames.append(rgb_grid)

# Display the video in the notebook
media.show_video(grid_frames, fps=30)
